# Data Preparation Pipeline (SwAIther-Precip)

This notebook documents and executes the full data-preparation pipeline used
to build the training datasets for **SwAIther-Precip**, producing the NetCDF
files consumed by the training and inference scripts.

## Pipeline overview

1. **Preprocess AIFS forecasts** — rename time coordinate, enforce
   non-negativity on precipitation channels, convert to mm/h.
2. **Preprocess CombiPrecip observations** — filter out near-zero-rain time
   steps, cap per-image outliers using a fitted gamma distribution, harmonize
   naming and coordinates.
3. **Build a coarse CombiPrecip target** — mass-conservative coarsening from
   the 1 km radar grid onto the AIFS native grid (used as the Step-1 target).
4. **Sanity check** — mass conservation between high-res and coarsened fields.
5. **Build multi-horizon datasets** — pair AIFS predictions with CombiPrecip
   targets at each lead time, plus single-horizon subsets (6 h, 1.5 d, 3 d,
   4.5 d, 6 d) for the per-lead specialist baselines.
6. **Visualization** — quick visual sanity check.
7. **Add elevation** — reproject the SwissTopo DHM25 digital elevation model
   onto each dataset's grid and append it as a static channel.
8. **Final assembly** — concatenate the 0–12 h and 12–24 h horizon ranges into
   a single 0–24 h dataset.

## Reproducing this notebook

Expected inputs in `../data/`:

- **Raw AIFS forecasts** as `AIFS_Single_pred_concat_2019_2023_0_12Horizons.nc`
  and `..._12_24Horizons.nc` (obtained from ECMWF; see
  `docs/data_preparation.md`).
- **CombiPrecip observations** as
  `CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_*.nc`
  (obtained from MeteoSwiss under their licensing terms — not redistributable).
- **SwissTopo DHM25** digital elevation model as `dhm25_grid_raster.asc`.

⚠️ **This notebook is provided for transparency and reproducibility**, not as
a turnkey tutorial. Several cells are I/O-heavy and were originally run once
in sequence on an HPC node. The output datasets produced here are the
`AIFS_Single_pred_concat_*`, `CPCH_processed_precipitation_*`, and
`*_wElevation.nc` files referenced throughout the training configs.


## Setup

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gamma

plt.rcParams["lines.linewidth"] = 2

### Utility functions

A small zero-proportion helper used to filter dry images, and a Switzerland-bordered
plotting helper used for the visualization section.

In [2]:
def Extract_Zero_proportion(ds, feature_name):
    """
    For each time step in `ds`, compute the fraction of pixels that are zero
    (and the fraction that are negative). Returns the dataset-wide zero
    proportion and per-time dataframes for both.
    """
    zero_mask = ds[feature_name] == 0
    negative_mask = ds[feature_name] < 0

    total_pixels = ds[feature_name].size
    zero_count = zero_mask.sum().item()
    negative_count = negative_mask.sum().item()
    zero_proportion = zero_count / total_pixels

    zero_per_time = zero_mask.mean(dim=["lat", "lon"])
    neg_per_time = negative_mask.mean(dim=["lat", "lon"])
    zero_proportion_df = zero_per_time.to_dataframe(name="zero_proportion")
    negative_proportion_df = neg_per_time.to_dataframe(name="negative_proportion")

    return zero_proportion, zero_proportion_df, negative_proportion_df

The `ColormapConfig` and `Plot_images_WithCHborders_indices` helpers below are also available in `helpers/viz_helpers.py`; they're inlined here so the notebook is self-contained.

In [3]:
from matplotlib import colors

class ColormapConfig:
    def __init__(self):
        self.cmap = None
        self.norm = None
        self.clevs = None

        self.build_colormap()

    def build_colormap(self):
        # Define the colormap boundaries and colors
        # color_list = ['lightgrey', 'lightskyblue', 'blue', 'yellow', 'orange', 'red', 'darkred']
        color_list = [
            "lightblue",
            "dodgerblue",
            "blue",
            "navy",
            "yellow",
            "orange",
            "green",
            "brown",
            "red",
        ]

        self.clevs = [0.000001, 0.01, 0.1, 0.5, 1.5, 2.5, 4, 6, 10]  # mm/hr

        # Create a ListedColormap object with the defined colors
        self.cmap = colors.ListedColormap(color_list)
        self.cmap.name = "Custom Colormap"

        # Set the color for values above the maximum level
        self.cmap.set_over("darkmagenta")
        # Set the color for values below the minimum level
        self.cmap.set_under("none")
        # Set the color for missing values
        self.cmap.set_bad("gray", alpha=0.5)

        # Create a BoundaryNorm object to normalize the data values to the colormap boundaries
        self.norm = colors.BoundaryNorm(self.clevs, self.cmap.N)


# Create an instance of the ColormapConfig class
cmap_custom = ColormapConfig()

In [ ]:
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader

def Plot_Some_images(ds, featureName):

    # 1) Instantiate your colormap/norm
    config = ColormapConfig()
    cmap_custom = config.cmap
    norm_custom = config.norm
    clevs = config.clevs

    # 2) Pick a few images
    n_images = 6
    n_times = ds.dims["time"]
    indices = np.linspace(0, n_times - 1, n_images, dtype=int)

    # 3) Make 2×3 grid
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
    axes = axes.flatten()

    for ax, idx in zip(axes, indices):
        da = ds[featureName].isel(time=idx)
        im = ax.imshow(
            da,  # 2D array
            cmap=cmap_custom,  # your ListedColormap
            norm=norm_custom,  # your BoundaryNorm
            origin="lower",  # or whatever geographic orientation you prefer
        )
        ts = np.datetime_as_string(ds.time.values[idx], unit="D")
        ax.set_title(f"Idx {idx} — {ts}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    # 4) Shared colorbar with boundaries at your clevs
    cbar = fig.colorbar(
        im,
        ax=axes,
        orientation="vertical",
        boundaries=clevs,
        ticks=clevs,
        extend="both",  # shows arrows for under/over
        shrink=0.7,
    )
    cbar.set_label("6-h Cumulative Precipitation (mm/hr)")

    plt.show()

    return fig


def Plot_images_WithCHborders(ds, featureName):
    # 1) Prepare your custom colormap/norm
    config = ColormapConfig()
    cmap_custom = config.cmap
    norm_custom = config.norm
    clevs = config.clevs

    # 2) Read the Natural Earth “admin_0_countries” shapefile
    shp_fname = shpreader.natural_earth(
        resolution="50m", category="cultural", name="admin_0_countries"
    )
    reader = shpreader.Reader(shp_fname)
    # Filter to the Switzerland feature
    switz_geom = [
        rec.geometry
        for rec in reader.records()
        if rec.attributes["NAME_LONG"] == "Switzerland"
    ][0]

    # 3) Pick a few indices to plot
    n_images = 6
    n_times = ds.dims["time"]
    indices = np.linspace(0, n_times - 1, n_images, dtype=int)

    # 4) Create a 2×3 grid with PlateCarree projection
    fig, axes = plt.subplots(
        2,
        3,
        figsize=(15, 8),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )
    axes = axes.flatten()

    for ax, idx in zip(axes, indices):
        da = ds[featureName].isel(time=idx)

        # 5) Plot the data
        im = ax.imshow(
            da,
            origin="lower",
            extent=[ds.lon.min(), ds.lon.max(), ds.lat.min(), ds.lat.max()],
            cmap=cmap_custom,
            norm=norm_custom,
            transform=ccrs.PlateCarree(),
        )

        # 6) Draw the Switzerland border on top
        ax.add_geometries(
            [switz_geom],
            crs=ccrs.PlateCarree(),
            facecolor="none",
            edgecolor="black",
            linewidth=1.2,
        )

        # 7) Tidy up
        ts = np.datetime_as_string(ds.time.values[idx], unit="D")
        ax.set_title(f"Idx {idx} — {ts}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    # 8) Shared colorbar
    cbar = fig.colorbar(
        im,
        ax=axes,
        boundaries=clevs,
        ticks=clevs,
        extend="both",
        shrink=0.7,
        orientation="vertical",
    )
    cbar.set_label("6-h Cumulative Precipitation (mm/hr)")

    plt.show()

    return fig


def Plot_images_WithCHborders_indices(
    ds: xr.Dataset,
    featureName: str,
    indices: list[int | str | np.datetime64] | None = None,
    title_custom="(Model X)",
    figsize=(15, 8),
    projection=ccrs.PlateCarree(),
):
    """
    Plot a grid of maps of `featureName` from `ds`, drawing Switzerland’s border on top.

    Parameters
    ----------
    ds : xarray.Dataset
      Must have dims ['time', 'lat', 'lon'].
    featureName : str
      The data variable to plot.
    indices : list of int or str or numpy.datetime64, optional
      Which time‐steps to plot. If None, picks 6 equally‐spaced frames.
    """
    # 1) custom cmap/norm
    cfg = ColormapConfig()
    cmap, norm, clevs = cfg.cmap, cfg.norm, cfg.clevs

    # 2) Switzerland geometry
    shp = shpreader.natural_earth("50m", "cultural", "admin_0_countries")
    rdr = shpreader.Reader(shp)
    switz = next(
        r.geometry for r in rdr.records() if r.attributes["NAME_LONG"] == "Switzerland"
    )

    # 3) Pick your indices
    n = ds.sizes["time"]
    time_index = pd.to_datetime(ds["time"].values)
    if indices is None:
        picks = list(np.linspace(0, n - 1, 6, dtype=int))
    else:
        picks = []
        for i in indices:
            if isinstance(i, (int, np.integer)):
                # clip
                picks.append(int(max(0, min(n - 1, i))))
            else:
                # interpret as datetime
                ts = pd.to_datetime(i)
                try:
                    loc = time_index.get_loc(ts)
                except KeyError:
                    # fall back to nearest
                    loc = time_index.get_indexer([ts], method="nearest")[0]
                    print(f"⚠️ {ts} not in index—using nearest at {time_index[loc]}")
                picks.append(int(loc))

    # unique & sorted
    picks = sorted(set(picks))
    N = len(picks)

    # 4) create grid (2×ceil(N/2))
    nrows = 2
    ncols = int(np.ceil(N / 2))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        subplot_kw={"projection": projection},
        constrained_layout=True,
    )
    axes = axes.flatten()

    # extent
    lon0, lon1 = float(ds.lon.min()), float(ds.lon.max())
    lat0, lat1 = float(ds.lat.min()), float(ds.lat.max())
    extent = [lon0, lon1, lat0, lat1]

    # 5) plot each
    for ax, idx in zip(axes, picks):
        da = ds[featureName].isel(time=idx)
        im = ax.imshow(
            da,
            origin="lower",
            extent=extent,
            cmap=cmap,
            norm=norm,
            transform=ccrs.PlateCarree(),
        )
        ax.add_geometries(
            [switz],
            crs=ccrs.PlateCarree(),
            facecolor="none",
            edgecolor="black",
            linewidth=1.2,
        )
        ts = np.datetime_as_string(ds.time.values[idx], unit="D")
        ax.set_title(f"Idx {idx} — {ts}" + title_custom, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    # hide unused
    for ax in axes[N:]:
        ax.set_visible(False)

    # 6) shared colorbar
    cbar = fig.colorbar(
        im,
        ax=axes[:N].tolist(),
        boundaries=clevs,
        ticks=clevs,
        extend="both",
        orientation="vertical",
        shrink=0.8,
        pad=0.02,
    )
    cbar.set_label(featureName)

    plt.show()

    return fig

## 1. Preprocess AIFS forecasts

AIFS forecasts are stored in two files split by lead-time range (0–12 h and
12–24 h). For each file we:
- Convert precipitation channels to mm/h (multiply by 1000, since the raw
  files are in m).
- Clip negative precipitation values (numerical artifacts) to zero.
- Rename the `prediction_time` coordinate to `time` for consistency.


In [ ]:
aifs_single_0_12 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_0_12Horizons.nc"
)
aifs_single_12_24 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_12_24Horizons.nc"
)

for ds in (aifs_single_0_12, aifs_single_12_24):
    ds["total_precipitation"] = ds["total_precipitation"] * 1000  # m -> mm
    ds["convective_precipitation"] = ds["convective_precipitation"] * 1000

    # Non-negative precipitation channels (drop tiny numerical negatives).
    ds["total_precipitation_NoNeg"] = ds["total_precipitation"].clip(min=0)
    ds["convective_precipitation_NoNeg"] = ds["convective_precipitation"].clip(min=0)

aifs_single_0_12 = aifs_single_0_12.rename({"prediction_time": "time"})
aifs_single_12_24 = aifs_single_12_24.rename({"prediction_time": "time"})

In [ ]:
aifs_single_0_12.to_netcdf(
    "../data/AIFS_Single_pred_concat_2019_2023_0_12Horizons_renamedTime_mm_NoNeg.nc",
    mode="w",
    format="NETCDF4",
)
aifs_single_12_24.to_netcdf(
    "../data/AIFS_Single_pred_concat_2019_2023_12_24Horizons_renamedTime_mm_NoNeg.nc",
    mode="w",
    format="NETCDF4",
)

## 2. Preprocess CombiPrecip observations

### 2.1 Filter dry time steps

Many CombiPrecip time steps contain essentially no rain (>99.5% of pixels = 0).
These dilute the training signal without adding information, so we drop them.


In [ ]:
precip0 = xr.open_dataset(
    "../data/CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_ValSampleStride4_6h_18hTest.nc"
)

zero_proportion_meteo, zero_proportion_df_meteo, _ = Extract_Zero_proportion(
    precip0, "precip6h_cumulative"
)
n_dropped = int((zero_proportion_df_meteo["zero_proportion"] >= 0.995).sum())
print(f"Dropping {n_dropped} time steps where >=99.5% of pixels are zero.")

In [ ]:
keep_mask = zero_proportion_meteo <= 0.995
meteoswiss_wet_enough = precip0.isel(time=keep_mask.values)
print(f"Kept {len(meteoswiss_wet_enough.time)} of {len(precip0.time)} time steps.")

### 2.2 Cap per-image outliers via a fitted gamma distribution

For each time step we fit a gamma distribution to the positive precipitation
values and cap pixels exceeding the 99.5%-percentile of that fit. This
attenuates the extreme upper tail (a few isolated radar artifacts per image)
without losing the meteorological signal.


In [ ]:
precip = meteoswiss_wet_enough["precip6h_cumulative"]
thresholds_995 = []

for da in precip:
    data = da.values.ravel()
    data = data[~np.isnan(data)]
    data_pos = data[data > 0]
    if len(data_pos) < 10:
        thresholds_995.append(np.nan)
        continue
    # Method-of-moments fit (fast and stable)
    mean, var = data_pos.mean(), data_pos.var()
    scale = var / mean
    shape = mean / scale
    thresholds_995.append(gamma.ppf(0.995, a=shape, scale=scale))

threshold_da = xr.DataArray(
    thresholds_995, coords=[meteoswiss_wet_enough["time"]], dims=["time"]
)
sns.displot(threshold_da)
plt.xlabel("Per-image 99.5% threshold (mm/6h)")
plt.show()

threshold_995_max = float(threshold_da.max())
print(f"Maximum per-image 99.5% threshold: {threshold_995_max:.1f} mm/6h")

In [ ]:
# Cap values above the maximum 99.5% threshold (single global cap).
meteoswiss_wet_enough = meteoswiss_wet_enough.chunk(
    {"time": 48, "lat": 256, "lon": 256}
)
meteoswiss_wet_enough["precip6h_cumulative"] = meteoswiss_wet_enough[
    "precip6h_cumulative"
].clip(max=threshold_995_max)

### 2.3 Final cleanup

- Rename the variable to reflect that it has been capped.
- Reorder latitude in ascending order (some files were stored descending).
- Take the mean over duplicated time stamps (a few 1-Jan values appear twice
  due to year-boundary preprocessing).


In [ ]:
meteoswiss_wet_enough = meteoswiss_wet_enough.rename_vars(
    {"precip6h_cumulative": "precip6h_cumulative_capped"}
)

# Reorder latitude ascending
meteoswiss_wet_enough = meteoswiss_wet_enough.reindex(
    lat=meteoswiss_wet_enough.lat[::-1]
)

# Average duplicated time stamps
meteoswiss_wet_enough = meteoswiss_wet_enough.groupby("time").mean()

In [ ]:
meteoswiss_wet_enough.to_netcdf(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma.nc",
    encoding={"precip6h_cumulative_capped": {"chunks": None}},
)

## 3. Build a coarse CombiPrecip target on the AIFS grid

Because CombiPrecip is a cumulative precipitation field, simple bilinear
interpolation onto the coarser AIFS grid does **not** preserve mass. We instead
apply a mass-conservative block-mean: for each coarse cell, we average the
high-res pixels whose centers fall inside its footprint.

The two helper functions below build the coarse-cell boundaries and perform
the aggregation in pure NumPy/xarray (no `xesmf` dependency).


In [ ]:
def bounds_from_centers(centers: np.ndarray) -> np.ndarray:
    """Given cell centers (length N), return N+1 cell boundaries."""
    centers = np.asarray(centers)
    mids = 0.5 * (centers[:-1] + centers[1:])
    first = centers[0] - (mids[0] - centers[0])
    last = centers[-1] + (centers[-1] - mids[-1])
    return np.concatenate([[first], mids, [last]])

In [ ]:
def aggregate_hr_to_lr_regular(
    da_hr: xr.DataArray,
    lat_lr: np.ndarray,
    lon_lr: np.ndarray,
    lat_name: str = "lat",
    lon_name: str = "lon",
):
    """
    Aggregate a high-res precip field on a strictly regular grid to a lower-res
    grid given by lat_lr, lon_lr (cell centers), assuming all high-res cells
    have equal area.

    Returns:
      da_lr_mean: coarse field (mean precip in each low-res cell, same units)
      cell_counts: number of high-res cells used for each low-res cell
    """
    # low-res boundaries
    lat_b = bounds_from_centers(lat_lr)
    lon_b = bounds_from_centers(lon_lr)

    # clip high-res to the low-res domain
    da_hr = da_hr.sel(
        {
            lat_name: slice(lat_b[0], lat_b[-1]),
            lon_name: slice(lon_b[0], lon_b[-1]),
        }
    )

    lat_hr = da_hr[lat_name].values
    lon_hr = da_hr[lon_name].values

    n_lat_lr = len(lat_lr)
    n_lon_lr = len(lon_lr)

    # keep other dims (e.g. time)
    other_dims = [d for d in da_hr.dims if d not in (lat_name, lon_name)]
    other_coords = {d: da_hr[d] for d in other_dims}
    other_shape = tuple(da_hr.sizes[d] for d in other_dims)

    da_lr_mean = xr.DataArray(
        np.full(other_shape + (n_lat_lr, n_lon_lr), np.nan, dtype=float),
        coords={**other_coords, lat_name: lat_lr, lon_name: lon_lr},
        dims=tuple(other_dims) + (lat_name, lon_name),
        name="precip6h_cumulative_capped",
        attrs=da_hr.attrs,
    )

    cell_counts = xr.DataArray(
        np.zeros((n_lat_lr, n_lon_lr), dtype=int),
        coords={lat_name: lat_lr, lon_name: lon_lr},
        dims=(lat_name, lon_name),
    )

    # loop over coarse cells (23*37 = 851, cheap)
    for i in range(n_lat_lr):
        # include right edge on the last cell
        if i < n_lat_lr - 1:
            lat_mask = (lat_hr >= lat_b[i]) & (lat_hr < lat_b[i + 1])
        else:
            lat_mask = (lat_hr >= lat_b[i]) & (lat_hr <= lat_b[i + 1])

        if not lat_mask.any():
            continue
        lat_sel = da_hr[lat_name].values[lat_mask]

        for j in range(n_lon_lr):
            if j < n_lon_lr - 1:
                lon_mask = (lon_hr >= lon_b[j]) & (lon_hr < lon_b[j + 1])
            else:
                lon_mask = (lon_hr >= lon_b[j]) & (lon_hr <= lon_b[j + 1])

            if not lon_mask.any():
                continue
            lon_sel = da_hr[lon_name].values[lon_mask]

            block = da_hr.sel({lat_name: lat_sel, lon_name: lon_sel})

            if block.sizes[lat_name] == 0 or block.sizes[lon_name] == 0:
                continue

            # high-res cells per coarse cell
            n_cells = block.sizes[lat_name] * block.sizes[lon_name]
            cell_counts[i, j] = n_cells

            # average high-res precip inside this coarse cell
            da_lr_mean.loc[{lat_name: lat_lr[i], lon_name: lon_lr[j]}] = block.mean(
                dim=(lat_name, lon_name)
            )

    return da_lr_mean, cell_counts


In [ ]:
# Target low-res grid (AIFS native)
low_lats = np.arange(43.75, 49.25 + 1e-6, 0.25)
low_lons = np.arange(3.25, 12.25 + 1e-6, 0.25)

precip_prep = xr.open_dataset(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma.nc"
)

precip_prep_low_correct, cell_counts = aggregate_hr_to_lr_regular(
    da_hr=precip_prep.precip6h_cumulative_capped,
    lat_lr=low_lats,
    lon_lr=low_lons,
)

precip_prep_low_correct.to_netcdf(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_lowRes_correct.nc",
    mode="w",
    format="NETCDF4",
)

## 4. Sanity check: mass conservation

Verify that the coarse target preserves total precipitation. We compare
`sum(high_res)` with `sum(mean_coarse * pixel_count_per_cell)` per time step.


In [ ]:
mass_hr = precip_prep.precip6h_cumulative_capped.sum(dim=["lat", "lon"])
mass_lr = (precip_prep_low_correct.precip6h_cumulative_capped * cell_counts).sum(
    dim=["lat", "lon"]
)

plt.figure(figsize=(6, 4))
sns.histplot(pd.Series((mass_hr - mass_lr).values), bins=100, stat="probability")
plt.xlabel("mass_hr - mass_lr (mm-pixels)")
plt.title("Per-time-step mass discrepancy after coarsening")
plt.show()

## 5. Build multi-horizon and single-horizon datasets

Merge the preprocessed AIFS forecasts with the coarse-resolution CombiPrecip
target along the common time axis. We produce both multi-horizon datasets
(0–12 h, 12–24 h) and single-horizon subsets used by the per-lead specialist
baselines.


In [ ]:
aifs_prep_0_12 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_0_12Horizons_renamedTime_mm_NoNeg.nc"
)
aifs_prep_12_24 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_12_24Horizons_renamedTime_mm_NoNeg.nc"
)

In [ ]:
# Common time axis between coarse precip and AIFS, separately for each horizon range
def merge_with_aifs(aifs_ds, precip_low_ds):
    common = np.intersect1d(precip_low_ds["time"].values, aifs_ds["time"].values)
    return xr.merge(
        [
            precip_low_ds.sel(time=common),
            aifs_ds.sel(time=common),
        ]
    )


ds_merged_0_12 = merge_with_aifs(aifs_prep_0_12, precip_prep_low_correct)
ds_merged_12_24 = merge_with_aifs(aifs_prep_12_24, precip_prep_low_correct)

ds_merged_0_12.to_netcdf(
    "../data/AIFS_Single_pred_concat_2019_2023_0_12Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    mode="w",
    format="NETCDF4",
)
ds_merged_12_24.to_netcdf(
    "../data/AIFS_Single_pred_concat_2019_2023_12_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    mode="w",
    format="NETCDF4",
)

### Single-horizon subsets

For the per-lead specialist baselines (UNet Single 6h, 3 day, 6 day, etc.),
extract single-lead subsets from the multi-horizon dataset.


In [ ]:
single_horizons = {
    "6h": ("0_12", 0),  # 6h lead
    "1d5": ("0_12", 5),  # 1.5 day lead
    "3day": ("0_12", 11),  # 3 day lead
    "4d5": ("12_24", 5),  # 4.5 day lead
    "6day": ("12_24", 11),  # 6 day lead
}

ds_by_range = {"0_12": ds_merged_0_12, "12_24": ds_merged_12_24}

for label, (rng, idx) in single_horizons.items():
    ds_single = ds_by_range[rng].sel(
        prediction_delta=ds_by_range[rng].prediction_delta[idx]
    )
    ds_single.to_netcdf(
        f"../data/AIFS_Single_pred_concat_2019_2023_SingleHori_{label}_NoNeg_andLowResPrecipWetEnough_gamma.nc",
        mode="w",
        format="NETCDF4",
    )

## 6. Visualization

Quick visual sanity check — high-res CombiPrecip alongside the coarse-resolution
target and the corresponding AIFS forecast at multiple lead times.


In [ ]:
hr_precip = xr.open_dataset(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_wElevation.nc"
)
aifs_data = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
)
aifs_6h = aifs_data.sel(prediction_delta=aifs_data.prediction_delta[0])
aifs_3days = aifs_data.sel(prediction_delta=aifs_data.prediction_delta[11])
aifs_6days = aifs_data.sel(prediction_delta=aifs_data.prediction_delta[23])

In [ ]:
date = "2020-09-20T00:00:00"

Plot_images_WithCHborders_indices(
    hr_precip,
    "precip6h_cumulative_capped",
    indices=[date],
    title_custom=" (CombiPrecip)",
)
Plot_images_WithCHborders_indices(
    aifs_6h,
    "total_precipitation_NoNeg",
    indices=[date],
    title_custom=" (AIFS, 6 h lead)",
)
Plot_images_WithCHborders_indices(
    aifs_3days,
    "total_precipitation_NoNeg",
    indices=[date],
    title_custom=" (AIFS, 3-day lead)",
)
Plot_images_WithCHborders_indices(
    aifs_6days,
    "total_precipitation_NoNeg",
    indices=[date],
    title_custom=" (AIFS, 6-day lead)",
)
plt.show()

## 7. Add elevation from the SwissTopo DHM

Reproject the SwissTopo DHM25 digital elevation model onto each dataset's grid
and append it as a static `altitude` channel. Used by the bias-correction
model as an invariant geographical input.

Requires `rioxarray` and `rasterio`:


In [ ]:
# !pip install rioxarray rasterio
import rioxarray as rxr

In [ ]:
def processDHM_AddToMeteoData(
    meteo_path: str,
    dem_path: str = "../data/dhm25_grid_raster.asc",
    as_data_var: bool = True,
) -> xr.Dataset:
    """
    Load a meteorological dataset and append a static `altitude` channel
    reprojected from the SwissTopo DHM25 raster onto the dataset's lat/lon grid.
    """

    # ----------------------------------------------------------
    # 1) Load meteo dataset
    # ------------------------------------------------------------
    # meteo_path = "../data/AIFS_Single_pred_concat_2019_2023_SingleHori_6h_NoNeg_andLowResPrecipWetEnough_gamma.nc"
    # meteo_path_HIGHres = "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma.nc"

    meteo = xr.open_dataset(meteo_path)

    # print("Meteo dims:", meteo.dims)
    # print("Meteo lat[0], lat[-1]:", float(meteo.lat[0]), float(meteo.lat[-1]))
    # print("Meteo lon[0], lon[-1]:", float(meteo.lon[0]), float(meteo.lon[-1]))

    # ------------------------------------------------------------
    # 2) Load DEM (ASCII DHM25)
    # ------------------------------------------------------------
    dem_path = "../data/dhm25_grid_raster.asc"
    dem = rxr.open_rasterio(dem_path, masked=True)  # shape: (1, y, x)

    # print("DEM dims:", dem.dims)
    # print("DEM CRS before:", dem.rio.crs)
    # print("DEM nodata before:", dem.rio.nodata)
    print("DEM bounds (native):", dem.rio.bounds())

    # ------------------------------------------------------------
    # 3) Attach correct CRS: CH1903 / LV03 (EPSG:21781)
    #    (your header xllcorner ~ 480000 -> LV03)
    # ------------------------------------------------------------
    dem = dem.rio.write_crs("EPSG:21781", inplace=False)
    # print("DEM CRS after:", dem.rio.crs)
    # print("DEM bounds (LV03):", dem.rio.bounds())

    # Ensure nodata is set and converted to NaN
    if dem.rio.nodata is None:
        dem = dem.rio.write_nodata(-9999, inplace=False)

    nodata_val = dem.rio.nodata
    if nodata_val is not None and not np.isnan(nodata_val):
        dem = dem.where(dem != nodata_val)

    # ------------------------------------------------------------
    # 4) Build template from meteo grid (lat/lon)
    #    This is the grid we want the DEM on.
    # ------------------------------------------------------------
    lat = meteo["lat"]
    lon = meteo["lon"]

    template = xr.DataArray(
        coords={"lat": lat, "lon": lon},
        dims=("lat", "lon"),
    )

    template = template.rio.write_crs(
        "EPSG:4326"
    ).rio.set_spatial_dims(  # meteo is in lat/lon
        x_dim="lon", y_dim="lat", inplace=False
    )

    # ------------------------------------------------------------
    # 5) Reproject DEM directly to the meteo grid
    # ------------------------------------------------------------
    dem_matched = dem.rio.reproject_match(template)

    # print("dem_matched dims:", dem_matched.dims)
    # print("dem_matched CRS:", dem_matched.rio.crs)
    # print("dem_matched nodata:", dem_matched.rio.nodata)

    # ------------------------------------------------------------
    # 6) Drop band, rename y/x -> lat/lon
    # ------------------------------------------------------------
    altitude = dem_matched.squeeze(drop=True)  # drop 'band' if present

    if "y" in altitude.dims and "x" in altitude.dims:
        altitude = altitude.rename({"y": "lat", "x": "lon"})

    print("altitude dims after rename:", altitude.dims)
    print(
        "altitude lat[0], lat[-1] before flip:",
        float(altitude.lat[0]),
        float(altitude.lat[-1]),
    )

    # ------------------------------------------------------------
    # 7) Ensure latitude orientation matches meteo
    #    (flip along lat only if directions differ)
    # ------------------------------------------------------------
    meteo_lat0 = float(meteo.lat[0])
    meteo_lat1 = float(meteo.lat[-1])
    alt_lat0 = float(altitude.lat[0])
    alt_lat1 = float(altitude.lat[-1])

    meteo_increasing = meteo_lat1 > meteo_lat0  # True if south -> north
    alt_increasing = alt_lat1 > alt_lat0

    if meteo_increasing != alt_increasing:
        # Flip altitude along latitude axis
        altitude = altitude.isel(lat=slice(None, None, -1))
        print("Flipped altitude along lat.")

    print("Meteo lat[0], lat[-1]:", meteo_lat0, meteo_lat1)
    print(
        "Altitude lat[0], lat[-1] after flip check:",
        float(altitude.lat[0]),
        float(altitude.lat[-1]),
    )

    # (Longitudes are usually fine, but you could do the same check for lon if needed.)

    # ------------------------------------------------------------
    # 8) Clean nodata again, just in case
    # ------------------------------------------------------------
    nodata_val = dem_matched.rio.nodata
    if nodata_val is not None:
        altitude = altitude.where(altitude != nodata_val)

    print("altitude.shape:", altitude.shape)
    print("altitude NaN fraction:", float(altitude.isnull().mean()))

    # Optional: quick sanity check on min/max height
    print(
        "altitude min/max (ignoring NaN):",
        float(altitude.min(skipna=True)),
        float(altitude.max(skipna=True)),
    )

    # ------------------------------------------------------------
    # 9) Attach altitude to meteo dataset
    # ------------------------------------------------------------
    meteo = meteo.assign_coords(altitude=altitude)

    return meteo
    # Optionally save
    # meteo.to_netcdf("AIFS_with_altitude.nc")

### Apply to all single-horizon datasets and the precip target

In [ ]:
# Process all single-horizon AIFS datasets (in-place re-save with elevation added)
for label in ["6h", "1d5", "3day", "4d5", "6day"]:
    path = f"../data/AIFS_Single_pred_concat_2019_2023_SingleHori_{label}_NoNeg_andLowResPrecipWetEnough_gamma.nc"
    ds = processDHM_AddToMeteoData(path)
    ds.to_netcdf(path, mode="w", format="NETCDF4")

# Add elevation to the high-res precip target as well
precip_path = "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma.nc"
precip_withElevation = processDHM_AddToMeteoData(precip_path)
precip_withElevation.to_netcdf(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_wElevation.nc",
    encoding={"precip6h_cumulative_capped": {"chunks": None}},
)

### Apply to the multi-horizon datasets

In [ ]:
for rng in ["0_12", "12_24"]:
    path = f"../data/AIFS_Single_pred_concat_2019_2023_{rng}Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
    ds = processDHM_AddToMeteoData(path)
    ds.to_netcdf(path, mode="w", format="NETCDF4")

## 8. Final assembly: concatenate horizon ranges into a single 0–24 h dataset

The two horizon-range files (0–12 h and 12–24 h) share the same time axis and
spatial grid; concatenate them along `prediction_delta` to get a single
0–24 h dataset (the canonical input for multi-lead training).


In [ ]:
ds_merged_0_12 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_0_12Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
)
ds_merged_12_24 = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_12_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
)

# The coarse precip target is the same in both, so drop it from the second to avoid conflict
ds_merged_12_24_no_precip = ds_merged_12_24.drop_vars("precip6h_cumulative_capped")

ds_full = xr.concat(
    [ds_merged_0_12, ds_merged_12_24_no_precip],
    dim="prediction_delta",
)
ds_full = ds_full.sortby("prediction_delta")

ds_full.to_netcdf(
    "../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc",
    mode="w",
    format="NETCDF4",
)

---

The output files in `../data/` are now ready for training. See
`docs/data_preparation.md` for a summary of the produced files and how they
map to the various training configurations in `conf/`.
